## Function 3: Calculate Station Statistics 📈

In this notebook, you'll learn how to build the `calculate_station_statistics()` function step by step. This is like taking many individual readings from each weather station and turning them into a clean station-by-station summary.

### 🎯 What This Function Does
- Groups environmental readings by station
- Calculates average temperature for each station
- Calculates average humidity for each station
- Counts how many readings each station has
- Returns a clean summary DataFrame for reporting and analysis

### 🔧 Function Signature
```python
def calculate_station_statistics(df):
    """
    Args:
        df (pandas.DataFrame): Environmental monitoring data
    
    Returns:
        pandas.DataFrame: Summary statistics by station
    """
```

### 📍 Where This Function Goes:
**Target File**: `src/pandas_basics.py`  
**Function Name**: `calculate_station_statistics()`  
**Replace**: The TODO comments with your working code

---

### 🚀 Step 1: Import Required Libraries

First, let's import the libraries we need:

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

print(f"✅ Pandas version: {pd.__version__}")
print("📊 Ready to calculate station statistics!")

### 🏗️ Step 2: Create Sample Environmental Monitoring Data

Like the original notebook, we'll use sample data to understand grouping and aggregation step by step:

In [ ]:
# Create realistic environmental monitoring data for demonstration
def create_sample_environmental_data():
    """Create sample environmental monitoring data that mimics real weather station data."""
    
    # Define weather stations with different characteristics
    stations = {
        'STN_001': {'base_temp': 22, 'base_humidity': 65, 'location': 'Downtown'},
        'STN_002': {'base_temp': 19, 'base_humidity': 72, 'location': 'Coastal'},
        'STN_003': {'base_temp': 25, 'base_humidity': 58, 'location': 'Desert'},
        'STN_004': {'base_temp': 16, 'base_humidity': 78, 'location': 'Mountain'},
        'STN_005': {'base_temp': 21, 'base_humidity': 68, 'location': 'Suburban'}
    }
    
    # Generate readings for each station
    all_data = []
    
    for station_id, props in stations.items():
        n_readings = np.random.randint(150, 300)
        
        base_temp = props['base_temp']
        temperatures = []
        
        for i in range(n_readings):
            hour_of_day = (i * 6) % 24
            daily_variation = 3 * np.sin(2 * np.pi * hour_of_day / 24)
            random_variation = np.random.normal(0, 2)
            temp = base_temp + daily_variation + random_variation
            temperatures.append(round(temp, 1))
        
        base_humidity = props['base_humidity']
        humidities = []
        
        for temp in temperatures:
            temp_effect = -0.8 * (temp - base_temp)
            random_variation = np.random.normal(0, 3)
            humidity = base_humidity + temp_effect + random_variation
            humidity = max(30, min(95, humidity))
            humidities.append(round(humidity, 1))
        
        for i in range(n_readings):
            all_data.append({
                'station_id': station_id,
                'temperature_c': temperatures[i],
                'humidity_percent': humidities[i],
                'location': props['location']
            })
    
    df = pd.DataFrame(all_data)
    df = df.sample(frac=1).reset_index(drop=True)
    
    return df

# Create our sample dataset
environmental_data = create_sample_environmental_data()

print("✅ Sample environmental monitoring data created!")
print(f"📊 Total readings: {len(environmental_data):,}")
print(f"📍 Stations: {environmental_data['station_id'].nunique()}")
print("\n🔍 First 5 rows:")
display(environmental_data.head())

print("\n📋 Readings per station:")
print(environmental_data['station_id'].value_counts().sort_index())

### 🔍 Step 3: Understand the Data Before Grouping

Before using `.groupby()`, let's see what stations are in the dataset and how many readings each one has:

In [ ]:
unique_stations = environmental_data['station_id'].unique()

print("🏷️ Unique stations:")
print(list(unique_stations))
print(f"\n📍 Number of stations: {len(unique_stations)}")

print("\n📊 Readings per station:")
print(environmental_data['station_id'].value_counts().sort_index())

### 🧠 Step 4: Learn How `.groupby()` Works

The `.groupby()` method splits the data into groups. Here, each group represents one station:

In [ ]:
# Create a groupby object
grouped = environmental_data.groupby('station_id')

print("🔧 GroupBy object created")
print(f"Type: {type(grouped)}")
print(f"Number of groups: {grouped.ngroups}")

print("\n📂 Group information:")
for name, group in grouped:
    print(f"   {name}: {len(group)} rows")
    if name == unique_stations[0]:
        print(f"\n🔍 Sample from {name}:")
        display(group[['temperature_c', 'humidity_percent', 'location']].head(3))
        break

### 📈 Step 5: Calculate Basic Statistics for Each Station

Now let's calculate the summary values we want for each station:

In [ ]:
# Average temperature by station
avg_temperature = grouped['temperature_c'].mean().round(1)
print("🌡️ Average temperature by station:")
print(avg_temperature)

print("\n" + "=" * 50 + "\n")

# Average humidity by station
avg_humidity = grouped['humidity_percent'].mean().round(1)
print("💧 Average humidity by station:")
print(avg_humidity)

print("\n" + "=" * 50 + "\n")

# Reading count by station
reading_count = grouped.size()
print("🔢 Reading count by station:")
print(reading_count)

### 🏗️ Step 6: Build a Summary DataFrame

Next, let's combine those separate results into one clean summary table:

In [ ]:
summary_df = pd.DataFrame({
    'station_id': avg_temperature.index,
    'avg_temperature': avg_temperature.values,
    'avg_humidity': avg_humidity.values,
    'reading_count': reading_count.values
})

print("📋 Station summary DataFrame:")
display(summary_df)

### 🔍 Step 7: Explore the Summary Results

Once the summary table is built, we can inspect it to find useful patterns:

In [ ]:
print("📊 Summary overview:")
print(f"Temperature range: {summary_df['avg_temperature'].min():.1f}°C to {summary_df['avg_temperature'].max():.1f}°C")
print(f"Humidity range: {summary_df['avg_humidity'].min():.1f}% to {summary_df['avg_humidity'].max():.1f}%")
print(f"Total readings represented: {summary_df['reading_count'].sum():,}")

hottest_station = summary_df.loc[summary_df['avg_temperature'].idxmax()]
coolest_station = summary_df.loc[summary_df['avg_temperature'].idxmin()]

print(f"\n🔥 Hottest station: {hottest_station['station_id']} ({hottest_station['avg_temperature']:.1f}°C)")
print(f"❄️ Coolest station: {coolest_station['station_id']} ({coolest_station['avg_temperature']:.1f}°C)")

### 🏗️ Step 8: Building the Complete Function

Now let's put everything together into a reusable function. This is what you will implement in `src/pandas_basics.py`:

Find this:
```python
def calculate_station_statistics(df):
    # TODO: Print header with function name
    # TODO: Validate input DataFrame (check if None or empty)
    # TODO: Check for required columns: ['station_id', 'temperature_c', 'humidity_percent']
    # TODO: Print summary of input data
    # TODO: Get unique stations and report count
    # TODO: Group data by station_id using df.groupby('station_id')
    # TODO: Calculate avg_temperature using .mean().round(1)
    # TODO: Calculate avg_humidity using .mean().round(1)
    # TODO: Count readings per station using .size()
    # TODO: Create summary DataFrame with all statistics
    # TODO: Reset index to make station_id a regular column
    # TODO: Print summary statistics (ranges, totals)
    # TODO: Find and report hottest/coolest stations
    # TODO: Return the summary DataFrame
```

Replace ALL the TODO comments with your working code from below

In [ ]:
def calculate_station_statistics_example(df):
    """Example implementation of the calculate_station_statistics function."""
    
    # Print header
    print("=" * 50)
    print("CALCULATING STATION STATISTICS")
    print("=" * 50)
    
    # Input validation
    if df is None or len(df) == 0:
        print("Error: DataFrame is empty or None")
        return pd.DataFrame()
    
    # Check for required columns
    required_columns = ['station_id', 'temperature_c', 'humidity_percent']
    missing_columns = [col for col in required_columns if col not in df.columns]
    
    if missing_columns:
        print(f"Error: Missing required columns: {missing_columns}")
        print(f"Available columns: {list(df.columns)}")
        return pd.DataFrame()
    
    # Print input data summary
    print(f"Processing {len(df):,} temperature readings...")
    
    # Get unique stations
    unique_stations = df['station_id'].unique()
    print(f"Found {len(unique_stations)} weather stations: {list(unique_stations)}")
    
    # Group data by station
    grouped = df.groupby('station_id')
    
    # Calculate statistics
    avg_temperature = grouped['temperature_c'].mean().round(1)
    avg_humidity = grouped['humidity_percent'].mean().round(1)
    reading_count = grouped.size()
    
    # Create summary DataFrame
    summary = pd.DataFrame({
        'station_id': avg_temperature.index,
        'avg_temperature': avg_temperature.values,
        'avg_humidity': avg_humidity.values,
        'reading_count': reading_count.values
    })
    
    # Reset index to make station_id a regular column
    summary = summary.reset_index(drop=True)
    
    # Print summary of results
    print(f"\nTemperature range across all stations: {summary['avg_temperature'].min():.1f}°C to {summary['avg_temperature'].max():.1f}°C")
    print(f"Humidity range across all stations: {summary['avg_humidity'].min():.1f}% to {summary['avg_humidity'].max():.1f}%")
    print(f"Total readings processed: {summary['reading_count'].sum():,}")
    print(f"Average readings per station: {summary['reading_count'].mean():.0f}")
    
    # Find temperature extremes
    hottest_station = summary.loc[summary['avg_temperature'].idxmax()]
    coolest_station = summary.loc[summary['avg_temperature'].idxmin()]
    
    print(f"\nHottest station: {hottest_station['station_id']} (avg: {hottest_station['avg_temperature']:.1f}°C)")
    print(f"Coolest station: {coolest_station['station_id']} (avg: {coolest_station['avg_temperature']:.1f}°C)")
    
    print("\nStation statistics calculated successfully!")
    
    return summary

### ✨ Step 9: Test Your Function with Sample Data

Let's first test our complete function with the sample environmental data from this notebook:

In [ ]:
print("🧪 TESTING WITH SAMPLE ENVIRONMENTAL DATA\n")
station_stats = calculate_station_statistics_example(environmental_data)

print("\n📋 Final results:")
display(station_stats)

### 🧪 Step 10: Test with the Temperature Readings CSV

Now let's test the same function using the course CSV file, just like the other notebooks:

In [ ]:
csv_file = '../../data/temperature_readings.csv'
csv_df = pd.read_csv(csv_file)

print(f"📁 Loaded CSV file: {csv_file}")
print(f"📊 Shape: {csv_df.shape}")
print(f"📋 Columns: {list(csv_df.columns)}")
print("\n🔍 First 5 rows:")
display(csv_df.head())

In [ ]:
print("🧪 TESTING WITH TEMPERATURE READINGS CSV\n")
csv_station_stats = calculate_station_statistics_example(csv_df)

print("\n📋 Final CSV results:")
display(csv_station_stats)

In [ ]:
# Test error handling with an empty DataFrame
print("\n" + "=" * 80 + "\n")
print("🧪 TESTING ERROR HANDLING - EMPTY DATAFRAME\n")
empty_df = pd.DataFrame()
result_empty = calculate_station_statistics_example(empty_df)
print(result_empty)

In [ ]:
# Test error handling with missing columns
print("\n" + "=" * 80 + "\n")
print("🧪 TESTING ERROR HANDLING - MISSING COLUMNS\n")
bad_df = environmental_data[['station_id', 'temperature_c']].copy()
result_bad = calculate_station_statistics_example(bad_df)
print(result_bad)

### 🧪 Step 11: Verify with Pytest

Test your entire function to verify your implementation in `src/pandas_basics.py` by running the following line in the terminal

```bash
uv run pytest tests/test_pandas_basics.py::TestCalculateStationStatistics -v

```

**⚠️ IMPORTANT: Make sure this passes before you move on!**

---

### 🔑 Key Learning Points

- **`df.groupby('station_id')`** splits data into groups by station
- **`.mean()`** calculates averages for each group
- **`.size()`** counts how many rows each group contains
- You can combine multiple grouped results into a new summary DataFrame
- **`.round(1)`** helps format values to one decimal place
- Always check that your required columns exist before processing
- Grouping and aggregation help turn raw data into useful summaries